In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "data" / "splits" / "train.json"
VAL_PATH = PROJECT_ROOT / "data" / "splits" / "validation.json"

CORE_CATEGORIES = [
    "Anti-Assignment",
    "Change Of Control",
    "Expiration Date",
    "Renewal Term",
    "License Grant",
    "Non-Transferable License",
    "Exclusivity",
    "Non-Compete",
    "Cap On Liability",
    "Uncapped Liability",
]

print("Train exists:", TRAIN_PATH.exists())
print("Validation exists:", VAL_PATH.exists())

Train exists: True
Validation exists: True


In [2]:
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print("Training contracts:", len(train_data["data"]))
print("Validation contracts:", len(val_data["data"]))

Training contracts: 326
Validation contracts: 82


In [3]:
def get_category(qa):
    return qa["id"].rsplit("__", 1)[-1].rsplit("_", 1)[0]


def build_contract_presence(split_data, categories):
    rows = []

    for contract in split_data["data"]:
        presence = {category: False for category in categories}

        for paragraph in contract["paragraphs"]:
            for qa in paragraph["qas"]:
                category = get_category(qa)

                if category in presence and not qa["is_impossible"]:
                    presence[category] = True

        rows.append({
            "title": contract["title"],
            **presence
        })

    return pd.DataFrame(rows)


train_presence = build_contract_presence(
    train_data,
    CORE_CATEGORIES
)

val_presence = build_contract_presence(
    val_data,
    CORE_CATEGORIES
)

In [4]:
eligibility_rows = []

for category in CORE_CATEGORIES:
    train_positive = int(train_presence[category].sum())
    train_negative = len(train_presence) - train_positive

    val_positive = int(val_presence[category].sum())
    val_negative = len(val_presence) - val_positive

    eligible = (
        train_positive > 0
        and train_negative > 0
        and val_positive > 0
        and val_negative > 0
    )

    eligibility_rows.append({
        "category": category,
        "train_positive": train_positive,
        "train_negative": train_negative,
        "validation_positive": val_positive,
        "validation_negative": val_negative,
        "eligible": eligible
    })

eligibility_df = pd.DataFrame(eligibility_rows)

eligibility_df

,category,train_positive,train_negative,validation_positive,validation_negative,eligible
0,Anti-Assignment,235,91,67,15,True
1,Change Of Control,76,250,19,63,True
2,Expiration Date,264,62,71,11,True
3,Renewal Term,119,207,31,51,True
4,License Grant,164,162,41,41,True
5,Non-Transferable License,90,236,26,56,True
6,Exclusivity,116,210,31,51,True
7,Non-Compete,77,249,19,63,True
8,Cap On Liability,181,145,50,32,True
9,Uncapped Liability,78,248,20,62,True


In [5]:
assert eligibility_df["eligible"].all(), (
    "At least one Core Category does not satisfy "
    "the Gate 3 coverage requirement."
)

print("All 10 proposed Core Categories pass the eligibility check.")

All 10 proposed Core Categories pass the eligibility check.


## Chunking Method Comparison

### Decision Criteria

Before comparing candidate chunking methods, we define the criteria that will
be used to evaluate them.

Required criteria:

1. **Complete Span Coverage**
   - All annotated spans must be represented by the chunking output.
   - No positive annotation may be lost.

2. **Boundary Splits**
   - Record how many annotated spans cross chunk boundaries.
   - Fewer boundary-split spans are preferred.

3. **Masked Labels**
   - If an annotated span only partially overlaps a chunk, that
     chunk-category pair must be masked rather than treated as positive
     or negative.
   - Compare how many masked pairs each chunking method produces.

4. **Overlap Cost**
   - Record how much additional chunk data is produced because of overlap.
   - More overlap may improve span coverage but creates additional data.

5. **Processing Cost**
   - Compare the number of chunks generated and the relative amount of
     processing required.

The final chunking method will be selected only after the candidate methods
are evaluated using these criteria.

### Candidate Chunking Methods

#### Candidate A: Fixed 512 / 128
- Maximum chunk length: 512 tokens
- Overlap: 128 tokens
- Step size: 384 tokens

#### Candidate B: Fixed 512 / 0
- Maximum chunk length: 512 tokens
- Overlap: 0 tokens
- Step size: 512 tokens

Candidate B provides a simple comparison that isolates the effect of overlap.

Neither candidate is considered approved until the comparison is complete
and the team records its decision.

### Tokenization Definition

The September milestone requires fixed 512-token chunks but does not specify
a tokenizer.

For this chunking comparison, a token is defined as one contiguous
non-whitespace sequence in the original contract text.

This definition is deterministic and preserves the original character
locations of every token, allowing chunk boundaries to be mapped directly
back to CUAD annotation spans.

The chunk-generation step uses source text only and does not inspect
annotations or labels.

In [6]:
import re
import hashlib


def tokenize_with_offsets(text):
    """
    Split text into non-whitespace tokens while preserving
    each token's character start/end positions.
    """
    tokens = []

    for match in re.finditer(r"\S+", text):
        tokens.append({
            "token": match.group(),
            "char_start": match.start(),
            "char_end": match.end()
        })

    return tokens

In [7]:
first_contract = train_data["data"][0]
first_text = first_contract["paragraphs"][0]["context"]

tokens = tokenize_with_offsets(first_text)

print("Contract:", first_contract["title"])
print("Characters:", len(first_text))
print("Tokens:", len(tokens))
print()
print(tokens[:10])

Contract: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Characters: 54290
Tokens: 5998

[{'token': 'EXHIBIT', 'char_start': 0, 'char_end': 7}, {'token': '10.6', 'char_start': 8, 'char_end': 12}, {'token': 'DISTRIBUTOR', 'char_start': 44, 'char_end': 55}, {'token': 'AGREEMENT', 'char_start': 56, 'char_end': 65}, {'token': 'THIS', 'char_start': 76, 'char_end': 80}, {'token': 'DISTRIBUTOR', 'char_start': 82, 'char_end': 93}, {'token': 'AGREEMENT', 'char_start': 95, 'char_end': 104}, {'token': '(the', 'char_start': 105, 'char_end': 109}, {'token': '"Agreement")', 'char_start': 111, 'char_end': 123}, {'token': 'is', 'char_start': 125, 'char_end': 127}]


In [8]:
def chunk_text(text, chunk_size=512, overlap=128):
    tokens = tokenize_with_offsets(text)

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    step = chunk_size - overlap
    chunks = []

    for chunk_index, token_start in enumerate(
        range(0, len(tokens), step)
    ):
        token_end = min(token_start + chunk_size, len(tokens))

        chunk_tokens = tokens[token_start:token_end]

        if not chunk_tokens:
            break

        char_start = chunk_tokens[0]["char_start"]
        char_end = chunk_tokens[-1]["char_end"]

        chunk_text_value = text[char_start:char_end]

        chunks.append({
            "chunk_index": chunk_index,
            "token_start": token_start,
            "token_end": token_end,
            "token_count": len(chunk_tokens),
            "char_start": char_start,
            "char_end": char_end,
            "text": chunk_text_value,
            "text_hash": hashlib.sha256(
                chunk_text_value.encode("utf-8")
            ).hexdigest()
        })

        if token_end == len(tokens):
            break

    return chunks

In [9]:
chunks_512_128 = chunk_text(
    first_text,
    chunk_size=512,
    overlap=128
)

print("Number of chunks:", len(chunks_512_128))

for chunk in chunks_512_128[:3]:
    print(
        chunk["chunk_index"],
        chunk["token_start"],
        chunk["token_end"],
        chunk["token_count"],
        chunk["char_start"],
        chunk["char_end"]
    )

Number of chunks: 16
0 0 512 512 0 4572
1 384 896 512 3263 8025
2 768 1280 512 7033 10636


In [14]:
CHUNK_CONFIG_VERSION = "gate3_chunking_v1"


def chunk_all_contracts(split_data, chunk_size, overlap, config_name):
    rows = []

    for contract in split_data["data"]:
        title = contract["title"]
        text = contract["paragraphs"][0]["context"]

        contract_chunks = chunk_text(
            text,
            chunk_size=chunk_size,
            overlap=overlap
        )

        for chunk in contract_chunks:
            chunk_id = (
                f"{title}__{config_name}__"
                f"chunk_{chunk['chunk_index']:04d}"
            )

            rows.append({
                "chunk_id": chunk_id,
                "title": title,
                "config": config_name,
                "config_version": CHUNK_CONFIG_VERSION,
                "chunk_size": chunk_size,
                "overlap": overlap,
                "step_size": chunk_size - overlap,
                "chunk_index": chunk["chunk_index"],
                "token_start": chunk["token_start"],
                "token_end": chunk["token_end"],
                "token_count": chunk["token_count"],
                "char_start": chunk["char_start"],
                "char_end": chunk["char_end"],
                "text": chunk["text"],
                "text_hash": chunk["text_hash"]
            })

    return pd.DataFrame(rows)

In [15]:
chunks_512_128 = chunk_all_contracts(
    train_data,
    chunk_size=512,
    overlap=128,
    config_name="512_128"
)

print("512/128 total chunks:", len(chunks_512_128))
print("Contracts:", chunks_512_128["title"].nunique())

512/128 total chunks: 6875
Contracts: 326


In [16]:
chunks_512_0 = chunk_all_contracts(
    train_data,
    chunk_size=512,
    overlap=0,
    config_name="512_0"
)

print("512/0 total chunks:", len(chunks_512_0))
print("Contracts:", chunks_512_0["title"].nunique())

512/0 total chunks: 5280
Contracts: 326


In [18]:
chunk_comparison = pd.DataFrame([
    {
        "method": "512/128",
        "total_chunks": len(chunks_512_128)
    },
    {
        "method": "512/0",
        "total_chunks": len(chunks_512_0)
    }
])

chunk_comparison

,method,total_chunks
0,512/128,6875
1,512/0,5280


In [19]:
def extract_core_spans(split_data, categories):
    rows = []

    for contract in split_data["data"]:
        title = contract["title"]

        for paragraph in contract["paragraphs"]:
            context = paragraph["context"]

            for qa in paragraph["qas"]:
                category = get_category(qa)

                if category not in categories:
                    continue

                for answer in qa.get("answers", []):
                    span_start = answer["answer_start"]
                    span_text = answer["text"]
                    span_end = span_start + len(span_text)

                    # Make sure CUAD's location really matches the text
                    assert context[span_start:span_end] == span_text

                    rows.append({
                        "title": title,
                        "category": category,
                        "span_start": span_start,
                        "span_end": span_end,
                        "span_text": span_text
                    })

    return pd.DataFrame(rows)


core_spans = extract_core_spans(
    train_data,
    CORE_CATEGORIES
)

print("Total core-category spans:", len(core_spans))
print("Contracts with core-category spans:", core_spans["title"].nunique())

core_spans.head()

Total core-category spans: 2695
Contracts with core-category spans: 296


,title,category,span_start,span_end,span_text
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,5268,5541,The term of this Agreement shall be ten (10)...
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Renewal Term,5543,5881,If Distributor comp...
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Exclusivity,1854,2004,The Company appoints the Distributor as an ex...
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Exclusivity,2112,2611,Company hereby appoints ...
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Exclusivity,12390,12503,The Distributor shall not order or ...


In [20]:
def build_chunk_labels(chunks_df, spans_df, categories):
    rows = []

    # Group spans so we can quickly find spans for a given
    # contract + category
    grouped_spans = {
        (title, category): group
        for (title, category), group
        in spans_df.groupby(["title", "category"])
    }

    for _, chunk in chunks_df.iterrows():
        chunk_start = chunk["char_start"]
        chunk_end = chunk["char_end"]

        for category in categories:
            relevant_spans = grouped_spans.get(
                (chunk["title"], category)
            )

            has_full_span = False
            has_partial_overlap = False

            if relevant_spans is not None:
                for _, span in relevant_spans.iterrows():
                    span_start = span["span_start"]
                    span_end = span["span_end"]

                    # Entire annotation is inside the chunk
                    fully_contained = (
                        span_start >= chunk_start
                        and span_end <= chunk_end
                    )

                    # Annotation and chunk share at least some text
                    overlaps = (
                        span_start < chunk_end
                        and span_end > chunk_start
                    )

                    if fully_contained:
                        has_full_span = True

                    elif overlaps:
                        has_partial_overlap = True

            # Positive takes priority if a complete span exists
            if has_full_span:
                target = 1
                mask = 0
                label = "positive"

            elif has_partial_overlap:
                target = 0
                mask = 1
                label = "masked"

            else:
                target = 0
                mask = 0
                label = "negative"

            rows.append({
                "chunk_id": chunk["chunk_id"],
                "title": chunk["title"],
                "config": chunk["config"],
                "chunk_index": chunk["chunk_index"],
                "category": category,
                "target": target,
                "mask": mask,
                "label": label
            })

    return pd.DataFrame(rows)

In [21]:
labels_512_128 = build_chunk_labels(
    chunks_512_128,
    core_spans,
    CORE_CATEGORIES
)

labels_512_0 = build_chunk_labels(
    chunks_512_0,
    core_spans,
    CORE_CATEGORIES
)

In [22]:
print("512/128 rows:", len(labels_512_128))
print("Expected:", len(chunks_512_128) * 10)

print()

print("512/0 rows:", len(labels_512_0))
print("Expected:", len(chunks_512_0) * 10)

512/128 rows: 68750
Expected: 68750

512/0 rows: 52800
Expected: 52800


In [23]:
print("512/128")
print(labels_512_128["label"].value_counts())

print()

print("512/0")
print(labels_512_0["label"].value_counts())

512/128
label
negative    65892
positive     2363
masked        495
Name: count, dtype: int64

512/0
label
negative    50625
positive     1784
masked        391
Name: count, dtype: int64


In [24]:
def evaluate_span_coverage(chunks_df, spans_df):
    rows = []

    # Group chunks by contract so we only compare a span
    # against chunks from the same contract.
    chunks_by_title = {
        title: group
        for title, group in chunks_df.groupby("title")
    }

    for _, span in spans_df.iterrows():
        contract_chunks = chunks_by_title[span["title"]]

        fully_covered = False
        has_partial_overlap = False

        for _, chunk in contract_chunks.iterrows():
            chunk_start = chunk["char_start"]
            chunk_end = chunk["char_end"]

            span_start = span["span_start"]
            span_end = span["span_end"]

            fully_contained = (
                span_start >= chunk_start
                and span_end <= chunk_end
            )

            overlaps = (
                span_start < chunk_end
                and span_end > chunk_start
            )

            if fully_contained:
                fully_covered = True

            elif overlaps:
                has_partial_overlap = True

        rows.append({
            "title": span["title"],
            "category": span["category"],
            "span_start": span["span_start"],
            "span_end": span["span_end"],
            "fully_covered": fully_covered,
            "boundary_split": has_partial_overlap,
            "lost_positive": not fully_covered
        })

    return pd.DataFrame(rows)

In [25]:
span_eval_512_128 = evaluate_span_coverage(
    chunks_512_128,
    core_spans
)

span_eval_512_0 = evaluate_span_coverage(
    chunks_512_0,
    core_spans
)

In [26]:
print("512/128")
print("Total spans:", len(span_eval_512_128))
print(
    "Fully covered:",
    span_eval_512_128["fully_covered"].sum()
)
print(
    "Boundary split:",
    span_eval_512_128["boundary_split"].sum()
)
print(
    "Lost positives:",
    span_eval_512_128["lost_positive"].sum()
)

print()

print("512/0")
print("Total spans:", len(span_eval_512_0))
print(
    "Fully covered:",
    span_eval_512_0["fully_covered"].sum()
)
print(
    "Boundary split:",
    span_eval_512_0["boundary_split"].sum()
)
print(
    "Lost positives:",
    span_eval_512_0["lost_positive"].sum()
)

512/128
Total spans: 2695
Fully covered: 2675
Boundary split: 740
Lost positives: 20

512/0
Total spans: 2695
Fully covered: 2403
Boundary split: 292
Lost positives: 292


In [27]:
lost_512_128 = span_eval_512_128[
    span_eval_512_128["lost_positive"]
].copy()

print("Lost spans:", len(lost_512_128))

lost_512_128[
    [
        "title",
        "category",
        "span_start",
        "span_end"
    ]
]

Lost spans: 20


,title,category,span_start,span_end
94,ZEBRATECHNOLOGIESCORP_04_16_2014-EX-10.1-INTEL...,License Grant,53155,54390
102,ZEBRATECHNOLOGIESCORP_04_16_2014-EX-10.1-INTEL...,Non-Transferable License,53155,54390
130,CORIOINC_07_20_2000-EX-10.5-LICENSE AND HOSTIN...,Cap On Liability,33580,34467
153,LOYALTYPOINTINC_11_16_2004-EX-10.2-RESELLER AG...,Cap On Liability,67875,69150
276,WPPPLC_04_30_2020-EX-4.28-SERVICE AGREEMENT,Non-Compete,55726,58506
366,BICYCLETHERAPEUTICSPLC_03_10_2020-EX-10.11-SER...,Non-Compete,37806,38979
632,ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGR...,Uncapped Liability,61481,62564
638,ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGR...,Cap On Liability,61481,62564
795,MRSFIELDSORIGINALCOOKIESINC_01_29_1998-EX-10-F...,Non-Compete,207947,209383
803,MRSFIELDSORIGINALCOOKIESINC_01_29_1998-EX-10-F...,Non-Compete,207748,209383


In [28]:
lost_512_128 = lost_512_128.merge(
    core_spans[
        [
            "title",
            "category",
            "span_start",
            "span_end",
            "span_text"
        ]
    ],
    on=[
        "title",
        "category",
        "span_start",
        "span_end"
    ],
    how="left"
)

lost_512_128["char_length"] = (
    lost_512_128["span_end"]
    - lost_512_128["span_start"]
)

lost_512_128["word_count"] = (
    lost_512_128["span_text"]
    .str.split()
    .str.len()
)

lost_512_128[
    [
        "category",
        "char_length",
        "word_count",
        "span_text"
    ]
].sort_values(
    "word_count",
    ascending=False
)

,category,char_length,word_count,span_text
4,Non-Compete,2780,479,The Executive agrees and undertakes with the C...
13,License Grant,2246,326,Skype hereby grants to Online BVI and the Comp...
14,Non-Transferable License,2246,326,Skype hereby grants to Online BVI and the Comp...
18,License Grant,2342,283,All sublicensing by Equifax and other members ...
19,License Grant,1601,258,In the event that termination is the result of...
10,Exclusivity,1388,236,"Subject to the terms contained herein, PAPA JO..."
9,Non-Compete,1635,225,Franchisee also acknowledges that Pretzel T...
12,Non-Compete,1330,207,"Each Party agrees that, for a period of five y..."
17,Cap On Liability,1258,205,"EXCEPT AS STATED ABOVE, THE PROVIDER'S OBLIGAT..."
8,Non-Compete,1436,199,Franchisee therefore agrees that during the ...


### Third candidate with more overlap

In [29]:
chunks_512_256 = chunk_all_contracts(
    train_data,
    chunk_size=512,
    overlap=256,
    config_name="512_256"
)

span_eval_512_256 = evaluate_span_coverage(
    chunks_512_256,
    core_spans
)

print("512/256")
print("Total chunks:", len(chunks_512_256))
print(
    "Fully covered:",
    span_eval_512_256["fully_covered"].sum()
)
print(
    "Boundary split:",
    span_eval_512_256["boundary_split"].sum()
)
print(
    "Lost positives:",
    span_eval_512_256["lost_positive"].sum()
)

512/256
Total chunks: 10076
Fully covered: 2693
Boundary split: 600
Lost positives: 2


In [30]:
chunks_512_384 = chunk_all_contracts(
    train_data,
    chunk_size=512,
    overlap=384,
    config_name="512_384"
)

span_eval_512_384 = evaluate_span_coverage(
    chunks_512_384,
    core_spans
)

print("512/384")
print("Total chunks:", len(chunks_512_384))
print(
    "Fully covered:",
    span_eval_512_384["fully_covered"].sum()
)
print(
    "Lost positives:",
    span_eval_512_384["lost_positive"].sum()
)

512/384
Total chunks: 19650
Fully covered: 2694
Lost positives: 1


In [31]:
lost_512_384 = span_eval_512_384[
    span_eval_512_384["lost_positive"]
].copy()

lost_512_384 = lost_512_384.merge(
    core_spans[
        [
            "title",
            "category",
            "span_start",
            "span_end",
            "span_text"
        ]
    ],
    on=[
        "title",
        "category",
        "span_start",
        "span_end"
    ],
    how="left"
)

lost_512_384["token_count"] = (
    lost_512_384["span_text"]
    .apply(lambda x: len(tokenize_with_offsets(x)))
)

lost_512_384[
    ["category", "token_count", "span_text"]
]

,category,token_count,span_text
0,Non-Compete,479,The Executive agrees and undertakes with the C...


In [32]:
chunks_768_512 = chunk_all_contracts(
    train_data,
    chunk_size=768,
    overlap=512,
    config_name="768_512"
)

span_eval_768_512 = evaluate_span_coverage(
    chunks_768_512,
    core_spans
)

print("768/512")
print("Total chunks:", len(chunks_768_512))
print(
    "Fully covered:",
    span_eval_768_512["fully_covered"].sum()
)
print(
    "Boundary split:",
    span_eval_768_512["boundary_split"].sum()
)
print(
    "Lost positives:",
    span_eval_768_512["lost_positive"].sum()
)

768/512
Total chunks: 9765
Fully covered: 2695
Boundary split: 596
Lost positives: 0


In [34]:
labels_768_512 = build_chunk_labels(
    chunks_768_512,
    core_spans,
    CORE_CATEGORIES
)

print(labels_768_512["label"].value_counts())

label
negative    91952
positive     4999
masked        699
Name: count, dtype: int64


In [37]:
comparison = pd.DataFrame([
    {
        "method": "512/0",
        "total_chunks": len(chunks_512_0),
        "fully_covered": span_eval_512_0["fully_covered"].sum(),
        "boundary_split": span_eval_512_0["boundary_split"].sum(),
        "lost_positives": span_eval_512_0["lost_positive"].sum(),
        "masked_pairs": (labels_512_0["label"] == "masked").sum()
    },
    {
        "method": "512/128",
        "total_chunks": len(chunks_512_128),
        "fully_covered": span_eval_512_128["fully_covered"].sum(),
        "boundary_split": span_eval_512_128["boundary_split"].sum(),
        "lost_positives": span_eval_512_128["lost_positive"].sum(),
        "masked_pairs": (labels_512_128["label"] == "masked").sum()
    },

    {
        "method": "768/512",
        "total_chunks": len(chunks_768_512),
        "fully_covered": span_eval_768_512["fully_covered"].sum(),
        "boundary_split": span_eval_768_512["boundary_split"].sum(),
        "lost_positives": span_eval_768_512["lost_positive"].sum(),
        "masked_pairs": (labels_768_512["label"] == "masked").sum()
    }
])

comparison

,method,total_chunks,fully_covered,boundary_split,lost_positives,masked_pairs
0,512/0,5280,2403,292,292,391
1,512/128,6875,2675,740,20,495
2,768/512,9765,2695,596,0,699


## Chunking Comparison Result

The required 512-token / 128-token-overlap candidate produced 6,875
training chunks and fully covered 2,675 of the 2,695 annotated spans in
the selected Core Categories. Twenty positive annotations were not fully
contained in any chunk.

Increasing overlap improved annotation coverage, but fixed 512-token
windows continued to lose at least one annotation. Inspection showed that
the remaining case was a 479-token Non-Compete annotation, which was
difficult to contain reliably within a 512-token window because of its
position relative to chunk boundaries.

A 768-token / 512-token-overlap candidate produced 9,765 chunks and fully
covered all 2,695 annotated spans, with zero lost positive annotations.
The method therefore provides complete Span coverage at the cost of
additional overlapping chunks and processing.

### Recommendation

Based on the training-data comparison, I recommend the 768-token /
512-token-overlap configuration because it is the tested candidate that
achieves complete annotation coverage and zero lost positive annotations.



In [38]:
train_chunks = chunk_all_contracts(
    train_data,
    chunk_size=768,
    overlap=512,
    config_name="768_512"
)

val_chunks = chunk_all_contracts(
    val_data,
    chunk_size=768,
    overlap=512,
    config_name="768_512"
)

print("Training chunks:", len(train_chunks))
print("Validation chunks:", len(val_chunks))

Training chunks: 9765
Validation chunks: 2481


In [39]:
val_core_spans = extract_core_spans(
    val_data,
    CORE_CATEGORIES
)

In [40]:
train_labels = build_chunk_labels(
    train_chunks,
    core_spans,
    CORE_CATEGORIES
)

val_labels = build_chunk_labels(
    val_chunks,
    val_core_spans,
    CORE_CATEGORIES
)

In [41]:
assert len(train_labels) == len(train_chunks) * 10
assert len(val_labels) == len(val_chunks) * 10

assert (
    train_labels.groupby("chunk_id").size() == 10
).all()

assert (
    val_labels.groupby("chunk_id").size() == 10
).all()

print("Every train and validation chunk has exactly 10 category rows.")

Every train and validation chunk has exactly 10 category rows.


In [42]:
train_span_eval = evaluate_span_coverage(
    train_chunks,
    core_spans
)

val_span_eval = evaluate_span_coverage(
    val_chunks,
    val_core_spans
)

print(
    "Train lost positives:",
    train_span_eval["lost_positive"].sum()
)

print(
    "Validation lost positives:",
    val_span_eval["lost_positive"].sum()
)

Train lost positives: 0
Validation lost positives: 0


In [43]:
print("Approved chunking configuration")
print("Chunk size: 768")
print("Overlap: 512")
print("Step size: 256")

print("\nTraining")
print("Chunks:", len(train_chunks))
print("Chunk-category rows:", len(train_labels))
print("Masked pairs:", train_labels["mask"].sum())

print("\nValidation")
print("Chunks:", len(val_chunks))
print("Chunk-category rows:", len(val_labels))
print("Masked pairs:", val_labels["mask"].sum())

Approved chunking configuration
Chunk size: 768
Overlap: 512
Step size: 256

Training
Chunks: 9765
Chunk-category rows: 97650
Masked pairs: 699

Validation
Chunks: 2481
Chunk-category rows: 24810
Masked pairs: 164


In [44]:
# Exactly 10 category rows per chunk
assert (train_labels.groupby("chunk_id").size() == 10).all()
assert (val_labels.groupby("chunk_id").size() == 10).all()

# target and mask are binary
assert set(train_labels["target"].unique()).issubset({0, 1})
assert set(train_labels["mask"].unique()).issubset({0, 1})

assert set(val_labels["target"].unique()).issubset({0, 1})
assert set(val_labels["mask"].unique()).issubset({0, 1})

# chunk IDs are unique
assert train_chunks["chunk_id"].is_unique
assert val_chunks["chunk_id"].is_unique

print("Final chunking integrity checks passed.")

Final chunking integrity checks passed.


In [45]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "gate3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_chunks.to_csv(
    OUTPUT_DIR / "train_chunks_768_512.csv",
    index=False
)

val_chunks.to_csv(
    OUTPUT_DIR / "validation_chunks_768_512.csv",
    index=False
)

train_labels.to_csv(
    OUTPUT_DIR / "train_chunk_labels_768_512.csv",
    index=False
)

val_labels.to_csv(
    OUTPUT_DIR / "validation_chunk_labels_768_512.csv",
    index=False
)

print("Gate 3 chunk artifacts exported.")

Gate 3 chunk artifacts exported.
